# Public Transport Network graph modelling using GTFS data

### Pre-requisites

In [7]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

from utils.imports import *

----

# Graph processing

### Load GTFS data

In [8]:
path_to_sqlite = DATA_DIR / "sqlite/ireland.sqlite"
g = load_gtfs(str(path_to_sqlite))

### Generate L-space

In [9]:
#Get available modes for the city
[mode_to_string(x) for x in g.get_modes()]

['Rail', 'Bus']

In [21]:
#Generate networkX graph
L=generate_graph(g,
               "Rail", # Chosen mode from previous list
                start_hour=5, # Consider trips from 5am...
                end_hour=24) #  ... to midnight

Considering trips between 2025-02-21 05:00:00+01:00 and 2025-02-21 23:59:59+01:00
Number of edges:  4164
Number of nodes:  2103


In [22]:
plot_graph(L, back_map="OSM")

#### Understanding nodes and edges
Each node has an ID and a dictionary of attributes, including the latitude and longitude (coordinates) and a given name.

An edge is a triplet (n1,n2,attrs), where n1 is the ID of the origin stop, n2 is the ID of the destination stop, and attrs is a dictionary with the attributes of the edge, including:
- duration_avg: the average time in seconds to travel the edge
- d: the length in meters of the edge
- n_vehicles: the total number of vehicles that pass through the edge in the studied period
- route_I_counts: same as n_vehicles but split by line_id

In [23]:
#An example node
list(L.nodes(data=True))[100]

(1279,
 {'lat': 50.27893,
  'lon': 5.909209,
  'name': 'Vielsalm',
  'original_ids': [1279]})

In [24]:
#An example edge
list(L.edges(data=True))[100]

(367,
 357,
 {'shape_id': {},
  'direction_id': {},
  'headsign': {'Alost': 44,
   'Brussels Airport-Zaventem': 17,
   'Eupen': 1,
   'Genk': 3,
   'Hasselt': 12,
   'Landen': 1,
   'Liège-Guillemins': 5,
   'Louvain': 4,
   'Schaerbeek': 3,
   'Termonde': 28,
   'Tongres': 2,
   'Welkenraedt': 10},
  'duration_avg': 91.84615384615384,
  'n_vehicles': 130,
  'd': 804,
  'route_I_counts': {134: 1,
   192: 3,
   195: 12,
   196: 1,
   313: 44,
   330: 16,
   332: 10,
   333: 1,
   337: 7,
   421: 1,
   422: 1,
   424: 8,
   533: 3,
   535: 10,
   537: 1,
   622: 1,
   728: 2,
   735: 1,
   771: 2,
   775: 5}})

### Automatic merge
In this step, all stops with the exact same name are merged if (and only if) they are not farther away than **delta** meters

In [25]:
L_merged=merge_stops_with_same_name(L,delta=200) #Same name and closer to 200m

Merged Aalter - Aalter
Merged Aalter - Aalter
Merged Aalter - Aalter
Merged Aalter - Aalter
Merged Aarschot - Aarschot
Merged Aarschot - Aarschot
Merged Aarschot - Aarschot
Merged Aarschot - Aarschot
Merged Aarschot - Aarschot
Merged Aarsele - Aarsele
Merged Aarsele - Aarsele
Merged Acren - Acren
Merged Acren - Acren
Merged Aiseau - Aiseau
Merged Aiseau - Aiseau
Merged Alken - Alken
Merged Alken - Alken
Merged Alost - Alost
Merged Alost - Alost
Merged Alost - Alost
Merged Alost - Alost
Merged Alost - Alost
Merged Alost - Alost
Merged Alost - Alost
Merged Alost Kerrebroek - Alost Kerrebroek
Merged Amay - Amay
Merged Amay - Amay
Merged Ampsin - Ampsin
Merged Ampsin - Ampsin
Merged Andenne - Andenne
Merged Andenne - Andenne
Merged Anderlecht - Anderlecht
Merged Anderlecht - Anderlecht
Merged Angleur - Angleur
Merged Angleur - Angleur
Merged Angleur - Angleur
Merged Angleur - Angleur
Merged Ans - Ans
Merged Ans - Ans
Merged Ans - Ans
Merged Ans - Ans
Merged Ans - Ans
Merged Anseremme - Ans

In [26]:
plot_graph(L_merged, back_map="OSM")

### Remove islands 
In this step, disconnected stops are removed from the graph. This should be a rare case, e.g., new stops that are currently being built.

In [30]:
check_islands(L_merged)

Found the following disconnected nodes: [2162, 1611, 1, 10, 2, 9, 8, 67, 24, 23, 22, 33, 21, 39, 34]


Delete these nodes? (y/n) y


Removed the following disconnected nodes: [2162, 1611, 1, 10, 2, 9, 8, 67, 24, 23, 22, 33, 21, 39, 34]


### Merge recommender
In this step possible nodes to merge are recommended and user needs to confirm (Y) or skip (N) the suggestion.
The suggestions are based on a combination of the similarity between the names of the stops (0-100) and on the proximity of the stops

In [31]:
#### 1st round ####
# Almost overlapping stops (20m) regardless of shared name
string_match=0 
stop_distance=20 
merge_recommender(L_merged,
                    string_match, 
                    stop_distance)

#### 2nd round ####
# Stops with at least 75% of similarity in names and closer to 500m
string_match=75 
stop_distance=500
merge_recommender(L_merged,
                    string_match, 
                    stop_distance)

In [33]:
plot_graph(L_merged, back_map="OSM")

### Manual merge
Click on two stops (holding shift) and then click on Merge button to merge stops. The last selected stop is merged into the first selected stop.

In [12]:
manual_merge(L_merged,
            jupyter_url="http://localhost:8888") #Change the port number if you're running in a different port

In [21]:
save_graph(L_merged,DATA_DIR/"pkl/ireland.pkl") # Change to desired path

### Sanity check
This function prints some useful information that might help further cleaning the graph

In [22]:
sanity_check(L_merged)

Checking self loops...
---
Checking links only on one direction...
Edge exists only in one direction:  Drogheda (Macbride)  (node 11513)  to Balbriggan  (node 3410) 
Edge exists only in one direction:  Dublin Connolly  (node 1795)  to Clongriffin  (node 1809) 
Edge exists only in one direction:  Dublin Connolly  (node 1795)  to Broombridge  (node 10435) 
Edge exists only in one direction:  Dublin Connolly  (node 1795)  to Maynooth  (node 4408) 
Edge exists only in one direction:  Limerick Junction  (node 8065)  to Templemore Station  (node 7991) 
Edge exists only in one direction:  Thurles  (node 12953)  to Limerick (Colbert)  (node 7774) 
Edge exists only in one direction:  Thurles  (node 12953)  to Charleville  (node 7378) 
Edge exists only in one direction:  Dublin Heuston  (node 1804)  to Limerick Junction  (node 8065) 
Edge exists only in one direction:  Dublin Heuston  (node 1804)  to Ballybrophy  (node 4728) 
Edge exists only in one direction:  Dublin Heuston  (node 1804)  to Mo

### Save L-space graph

In [23]:
save_graph(L_merged,DATA_DIR/"pkl/ireland.pkl") # Change to desired path

----

# Assignment
Now that you have your cleaned L-space graph....

### Parameters
Update the following parameters based on the city you chosed

In [5]:
L_space_path= DATA_DIR/"pkl/ireland_cleaned.pkl"  # Path where the clean L-space graph was stored

### Load GTFS data

In [6]:
g=load_gtfs(str(str(DATA_DIR / "sqlite/ireland.sqlite")))

#Get available modes for the city
[mode_to_string(x) for x in g.get_modes()]

['Bus', 'Tram', 'Rail']

### Load L-space graph

In [7]:
L_graph=load_graph(L_space_path)
plot_graph(L_graph, back_map="OSM")

### Create P-space 

In [10]:
P_graph=P_space_v4(g,
          L_graph,
          start_hour=5, # Same as when building L-space
          end_hour=24, # Same as when building L-space
          mode="Rail") # Same as when building L-space

plot_graph(P_graph, back_map="OSM")

In [9]:
save_graph(P_graph, DATA_DIR / "pkl/ireland_P.pkl") # Change to desired path

### Compute shortests paths between all pairs of nodes.
#### Note: this may take several minutes (depending on the size of the graph)
For a given pair of nodes $i$ and $j$, This function computes $sp_1,\ldots, sp_m$, the $m$ shortest paths in $\mathbf{L}$-space, i.e., the shortest in terms of in-vehicle travel time, for each pair of nodes $i$ and $j$

The GTC between pairs of stops accounts for initial and transfer waiting times, in-vehicle travel times, and a time-equivalent penalty cost for transfers.

For each path in L-space, we compute the sum of the waiting times for each leg in the path and the number of transfers, according to the labels and number of hops of the corresponding paths in $\mathbf{P}$-space. 
The GTC is determined by the following weighted sum:

$  \text{GTC}(sp_i)= {\text{in_vehicle_time}(sp_i) + \mathbf{\alpha} \times \text{waiting_time}(sp_i) + \sum_{j=1}^{{\text{n_transfers}}(sp_i)}{\beta_J}}$

In [6]:
# Waiting penalty (multiplier)
alpha = 2 # Waiting time is multiplied by 2

# Transfer penalties in minutes. Arbitrary size of at least one element, with the minutes to add
# for the first, second, third... transfers. 
betas = [5,15] # 5 minutes for the first transfer, 15 minutes for any transfer after that.

# Number of shortest paths to retrieve
m = 3

gtc = get_all_GTC_v4(L_graph, P, m, alpha, betas)

#### Understanding the results

For each pair of nodes we get as many paths as requested (or as many as available if there are fewer alternative paths).

The results is a dictionary, indexed by the nodes of origin (first index) and the nodes of destination (second index). For each pair of nodes we get a list, with the alternative paths ordered by GTC.

Each path is again modeled as a dictionary, including the following attributes:
- path: the sequence of nodes in the path computed in the L-space graph
- GTC: the GTC corresponding to that path
- in_vehicle: the in-vehicle travel time in minutes of that path
- waiting_time: the waiting-time in minutes of that path (from P-space)
- n_transfers: the number of transfers of that path (hops in P-space)

In [7]:
#Example: check the GTC between nodes 51 and 37
gtc[51][37]

{'path': [51, 27, 25, 26, 16, 17, 15, 12, 14, 13, 10, 9, 36, 37],
 'GTC': 844,
 'in_vehicle': 140,
 'waiting_time': 342,
 'n_transfers': 2,
 'traveled_distance': 180963}

### Auxiliary functions
Here are some auxiliary functions that might be useful for all your assignments. Use them if you need to.

In [8]:
#Return the average waiting time (in minutes) per line and direction
average_waiting_time_per_line_per_direction(P)

{98: {'1': 49.56521739130435, '0': 45.96774193548387},
 113: {'1': 48.35431654676259, '0': 91.5972894482091},
 106: {'0': 51.16387592643426, '1': 66.05663345568956},
 110: {'0': 32.21425558835934, '1': 36.69642857142857},
 111: {'0': 38.08269858541893, '1': 27.890736342042754},
 114: {'0': 4.561486431423682, '1': 4.559718861264816},
 109: {'0': 87.94285714285715, '1': 102.421875},
 99: {'0': 138.9075630252101, '1': 99.89690721649485},
 100: {'0': 141.61490683229815, '1': 127.15384615384616},
 112: {'0': 14.022606382978724, '1': 20.766331658291456},
 101: {'0': 220.64516129032256, '1': 51.81818181818181},
 102: {'0': 144.1011235955056, '1': 151.77514792899407},
 103: {'1': 190.0, '0': 570.0},
 105: {'0': 80.22788203753352, '1': 78.13315926892952},
 104: {'1': 386.1290322580645, '0': 570.0},
 107: {'0': 85.64766839378238, '1': 95.40772532188842},
 108: {'0': 181.64835164835165, '1': 159.16981132075475}}

In [9]:
#Return the average speed (in km/h) for the whole network
average_speed_network(L_graph)

57.392172930440374

In [14]:
#Get dataframe with all events
df=get_events(g,
              mode="Rail",
              start_hour=5,
              end_hour=24)
df.head()

Considering trips between 2025-09-08 05:00:00+01:00 and 2025-09-08 23:59:59+01:00


,from_stop_I,to_stop_I,dep_time_ut,arr_time_ut,shape_id,direction_id,headsign,route_type,route_id,trip_I,duration,from_seq,to_seq,route_I
0,3409,10852,1757335500,1757335620,3-4452_10,1,Bray (Daly),2,3-4452_86289,3898,120,1,2,114
1,10852,3420,1757335680,1757335800,3-4452_10,1,Bray (Daly),2,3-4452_86289,3898,120,2,3,114
2,3420,3413,1757335800,1757335980,3-4452_10,1,Bray (Daly),2,3-4452_86289,3898,180,3,4,114
3,3413,1810,1757335980,1757336100,3-4452_10,1,Bray (Daly),2,3-4452_86289,3898,120,4,5,114
4,1810,1801,1757336100,1757336220,3-4452_10,1,Bray (Daly),2,3-4452_86289,3898,120,5,6,114


In [15]:
#Set a new attribute to edges
import networkx as nx
nx.set_edge_attributes(L_graph, 0, 'capacity') # Set attribute "capacity" with default value 0

In [16]:
#Plot the network and color nodes based on an attribute
plot_graph(L_graph, back_map="OSM", color_by="lat")

In [18]:
#Plot the network and color edges based on an attribute
plot_graph(L_graph, back_map="OSM", edge_color_by="duration_avg")